# DC Motor Generation

This notebook is the DC motor equivalent of `TestPendulumGeneration.ipynb`.

Instead of generating an `n`-link pendulum with symbolic mechanics, we use the autonomous no-input DC motor dataset added in `datasets/dc_motor.py`.

State:

```text
x = [omega, current]
```

Default equations:

```text
d omega / dt = (Kt * current - b * omega - Tl) / J
d current / dt = (Va - R * current - Ke * omega) / L
```

By default `Va=0` and `Tl=0`, so the motor is an autonomous dissipative system with a stable equilibrium at the origin.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from datasets.dc_motor import DEFAULTS, build, dc_motor_gradient


## Parameters and right-hand side

The helper `dc_motor_gradient(params)` returns a vectorized function `f(X)` that maps states to derivatives.


In [ ]:
params = DEFAULTS.copy()
params.update({
    "Va": 0.0,
    "Tl": 0.0,
})

rhs = dc_motor_gradient(params)

print("DC motor parameters")
for k, v in params.items():
    print(f"{k:>12s}: {v}")

print("\nDerivative at the origin:", rhs(np.array([0.0, 0.0], dtype=np.float32)))


## Linear system matrix

Because this no-input DC motor model is linear, we can write it as:

```text
x_dot = A x
```

where `x = [omega, current]`.


In [ ]:
J, b = params["J"], params["b"]
Kt, Ke = params["Kt"], params["Ke"]
R, L = params["R"], params["L"]

A = np.array([
    [-b / J,   Kt / J],
    [-Ke / L, -R / L],
], dtype=np.float64)

eigvals = np.linalg.eigvals(A)

print("A =")
print(A)
print("\nEigenvalues:", eigvals)
print("Stable linear system:", np.all(np.real(eigvals) < 0))


## Generate train/test datasets

This calls the same dataset builder that `train.py` uses through `DynamicLoad("datasets")`.

The returned tensors are:

```text
X: sampled states
Y: true derivatives f(X)
```


In [ ]:
train_ds = build({"n": "5000", "nocache": True})
test_ds = build({"n": "1000", "test": True, "nocache": True})

X_train, Y_train = train_ds.tensors
X_test, Y_test = test_ds.tensors

print("train X:", tuple(X_train.shape), "train Y:", tuple(Y_train.shape))
print("test  X:", tuple(X_test.shape), "test  Y:", tuple(Y_test.shape))

print("\nFirst five training samples:")
for x, y in zip(X_train[:5].numpy(), Y_train[:5].numpy()):
    print(f"x={x}, f(x)={y}")


## Vector field

This shows the learned target field: for every point `(omega, current)`, the arrow is `f(x)`.


In [ ]:
omega_grid = np.linspace(-5.0, 5.0, 25)
current_grid = np.linspace(-5.0, 5.0, 25)
Omega, Current = np.meshgrid(omega_grid, current_grid)

grid_states = np.stack([Omega.ravel(), Current.ravel()], axis=1).astype(np.float32)
grid_derivatives = rhs(grid_states)

dOmega = grid_derivatives[:, 0].reshape(Omega.shape)
dCurrent = grid_derivatives[:, 1].reshape(Current.shape)

speed = np.sqrt(dOmega**2 + dCurrent**2)
dOmega_n = dOmega / np.maximum(speed, 1e-8)
dCurrent_n = dCurrent / np.maximum(speed, 1e-8)

plt.figure(figsize=(7, 6))
plt.quiver(Omega, Current, dOmega_n, dCurrent_n, speed)
plt.scatter([0], [0], marker="x", s=80)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("DC motor vector field: x_dot = f(x)")
plt.colorbar(label="||f(x)||")
plt.tight_layout()
plt.show()


## RK4 simulation

The pendulum notebook integrates physical equations over time. Here we do the same with a fourth-order Runge-Kutta step.


In [ ]:
def rk4_step(rhs, x, dt):
    k1 = rhs(x)
    k2 = rhs(x + 0.5 * dt * k1)
    k3 = rhs(x + 0.5 * dt * k2)
    k4 = rhs(x + dt * k3)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def simulate(rhs, x0, steps=300, dt=0.01):
    x = np.asarray(x0, dtype=np.float32)
    trajectory = np.zeros((steps, *x.shape), dtype=np.float32)
    trajectory[0] = x
    for step in range(1, steps):
        x = rk4_step(rhs, x, dt).astype(np.float32)
        trajectory[step] = x
    return trajectory

initial_states = np.array([
    [ 4.0,  4.0],
    [ 4.0, -4.0],
    [-4.0,  4.0],
    [-4.0, -4.0],
    [ 2.0,  0.0],
    [ 0.0,  2.0],
], dtype=np.float32)

dt = 0.01
steps = 400
trajectories = simulate(rhs, initial_states, steps=steps, dt=dt)
time = np.arange(steps) * dt

print("trajectories shape:", trajectories.shape)


## State traces

Both `omega` and `current` should decay toward zero because the default system is no-input and dissipative.


In [ ]:
plt.figure(figsize=(8, 4))
for idx in range(trajectories.shape[1]):
    plt.plot(time, trajectories[:, idx, 0], label=f"path {idx}")
plt.xlabel("time [s]")
plt.ylabel("omega [rad/s]")
plt.title("Angular velocity trajectories")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
for idx in range(trajectories.shape[1]):
    plt.plot(time, trajectories[:, idx, 1], label=f"path {idx}")
plt.xlabel("time [s]")
plt.ylabel("current [A]")
plt.title("Current trajectories")
plt.tight_layout()
plt.show()


## Phase portrait

This is the DC motor analogue of plotting pendulum trajectories. The trajectories should spiral/curve into the origin depending on parameters.


In [ ]:
plt.figure(figsize=(6, 6))
for idx in range(trajectories.shape[1]):
    plt.plot(trajectories[:, idx, 0], trajectories[:, idx, 1], label=f"path {idx}")
    plt.scatter(trajectories[0, idx, 0], trajectories[0, idx, 1], marker="o")
plt.scatter([0], [0], marker="x", s=100)
plt.xlabel("omega [rad/s]")
plt.ylabel("current [A]")
plt.title("DC motor phase portrait")
plt.legend()
plt.tight_layout()
plt.show()
